In [ ]:
import pandas as pd
#reemplazar con la ruta de los archivos correcta
path_files="C:\\Users\\x286384\\OneDrive - MerckGroup\\Desktop\\Planning Prototypes\\PlanningTranslation\\"
FCST=pd.read_excel(path_files+"FCST Merck Abril 26.xlsx",sheet_name="Abril 2026")
PROD_DC_2=pd.read_excel(path_files+"PRODUCTOS DC 2.xlsx",sheet_name="LISTA PT-GRANEL")

In [ ]:
"""""
Module: P&G Forecast extraction 
Purpose: extraer ## de jeringas para el ems por producto
Date: 23/07/2026
Author: J.Gonzalez
"""
logbalanceo=[]
#Columnas importantes
id_cols = ['Market', 'SKUMERCK', 'SKUPROCTER', 'Description', 'Units']

#Calculo de mes y año actual
Month_today=pd.to_datetime("today").month
Year_today=pd.to_datetime("today").year
demand_today=pd.to_datetime(str(Year_today) + "-01-" + str(Month_today), format="%Y-%m-%d")
demand_today=str(demand_today)[0:10]

#extraccion columnas de demanda del mes actual
matching_cols = []
for col in FCST.columns:
    parsed = pd.to_datetime(col, errors='coerce')
    if pd.notna(parsed) and parsed.month == Month_today and parsed.year == Year_today:
        matching_cols.append(col)

#en caso de encontrar dos columnas con la misma fecha conserva la segunda que ya incluye el calculo total de piezas.
if len(matching_cols) >= 2:
    matching_cols = [matching_cols[-1]]

#Union de columnas id con columna demanda mes actual
if matching_cols:
    keep_cols = id_cols + matching_cols
    FCST_month = FCST[keep_cols].copy()
else:
    FCST_month = FCST[id_cols].copy()
FCST_month = FCST_month.rename(columns={FCST_month.columns[-1]: 'Demanda'})

#Forecast de jeringas por mes actual en csv
FCST_month.to_csv("FCST_month.csv", index=False)

In [ ]:
"""""
Module: Produccion por linea
Purpose: Separa los productos por linea que utiliza
Date: 23/07/2026
Author: J.Gonzalez
"""
#reduccion de columnas
PROD_DC_2=PROD_DC_2[['Market', 'SKUMERCK', 'Description','Linea Granel 1', 'Linea Granel 2']]

#union de jeringas por mes con su linea 
PROD_DC_2 = PROD_DC_2.merge(FCST_month[['SKUMERCK', 'Demanda', 'Units']], on='SKUMERCK', how='left')

#extraccion de prodctos procesables  en DC2
DC2=PROD_DC_2[['Market', 'SKUMERCK', 'Description','Linea Granel 2','Units','Demanda']]
DC2 = DC2[DC2['Linea Granel 2'].notnull()].copy()
DC2 = DC2.drop(columns=['Linea Granel 2'])

#eliminacion de prodctos procesables  en DC2
if PROD_DC_2['Linea Granel 2'].notnull().any():
    PROD_DC_2 = PROD_DC_2[PROD_DC_2['Linea Granel 2'].isnull()].copy()

#creacion DB exclusiva para DC1
DC1=PROD_DC_2[['Market', 'SKUMERCK', 'Description','Units','Demanda']]

#Forecast de jeringas por linea en csv
DC2.to_csv("DC2.csv", index=False)
DC1.to_csv("DC1.csv", index=False)

In [ ]:
"""""
Module: Balanceo sencillo
Purpose: reporta lotes minimos necesarios, y realiza un balanceo sencillo 
Date: 23/07/2026
Author: J.Gonzalez
"""
#Primer reporte es Generado como punto de comparación

# calculo de total de jeringas
TOTAL_DC1=DC1['Demanda'].sum()
TOTAL_DC2=DC2['Demanda'].sum()
TOTAL_DEMANDA=TOTAL_DC1+TOTAL_DC2

#calculo de lotes necesarios
LOTES_DC1=TOTAL_DC1/105200
LOTES_DC2=TOTAL_DC2/105200
lotes_necesarios=TOTAL_DEMANDA/105200

#calculo de residuos en jeringas
residuo_DC1=(1-(TOTAL_DC1/105200)%1)*105200
if residuo_DC1 == 105200.0:
    residuo_DC1 = 0
residuo_DC2=(1-(TOTAL_DC2/105200)%1)*105200
if residuo_DC2 == 105200.0:
    residuo_DC2=0
residuo_total=(1-(TOTAL_DEMANDA/105200)%1)*105200

# residuo directo de DC2 sin contar pedaceria de sobra
residuo_norm_dc2=((TOTAL_DC2/105200)%1)*105200


#generación de reporte 
print("Reporte situación actual")
print("TOTAL_DC1: "+str(TOTAL_DC1)+" total jeringas")
print("TOTAL_DC2: "+str(TOTAL_DC2)+" total jeringas")
print("TOTAL_DEMANDA: "+str(TOTAL_DEMANDA)+" total jeringas")
print("Lotes DC1 "+str(LOTES_DC1))
print("Lotes DC2 "+str(LOTES_DC2))
print("Total Demanda Lotes "+str(lotes_necesarios))
print("Residuo DC1 "+str(residuo_DC1)+" jeringas de sobra")
print("Residuo DC2 "+str(residuo_DC2)+" jeringas de sobra")
print("residuo de la demanda total "+str(residuo_DC1+residuo_DC2)+" jeringas de sobra")
print("Residuo Total "+str(residuo_total)+" jeringas de sobra")
print("")
print("")

"""
Aqui empieza el balanceo, solo busca ajustar para que la entrada de lotes a DC2 sea siempre un numero entero y mandar la pedaceria a DC1
"""
## busca el lote mas chico que pueda cumplir con las condiciones de ser menor que el residuo para balancear DC2
min_idx = DC2[(DC2['Demanda'] > residuo_norm_dc2) & (DC2['Units'] == 1)]['Demanda'].idxmin()
#reporte de linea antes de cambio
print("DC2:Fila antes de ajuste")
print(DC2.loc[[min_idx]])

# Se resta el residuo de la fila para balancear DC2
DC2.loc[min_idx, 'Demanda'] = DC2.loc[min_idx, 'Demanda'] - residuo_norm_dc2
print("DC2:Fila después de ajuste:")
print(DC2.loc[[min_idx]])

# Crea una nueva fila en DC1 con el residuo de DC2
row_to_copy = DC2.loc[[min_idx]]
row_to_copy['Demanda'] = residuo_norm_dc2
DC1 = pd.concat([DC1, row_to_copy], ignore_index=True)
print("DC1:Fila agregada:")
print(DC1.loc[[len(DC1)-1]])
print("")
#Log de cambios a DC1
if 'logbalanceo' not in globals():
    logbalanceo = []
else:
    logbalanceo.append(row_to_copy)

"""""
Genera el mismo reporte pero después del balanceo. 
"""

# calculo de total de jeringas
TOTAL_DC1=DC1['Demanda'].sum()
TOTAL_DC2=DC2['Demanda'].sum()
TOTAL_DEMANDA=TOTAL_DC1+TOTAL_DC2
#calculo de lotes necesarios
LOTES_DC1=TOTAL_DC1/105200
LOTES_DC2=TOTAL_DC2/105200
lotes_necesarios=TOTAL_DEMANDA/105200
#calculo de residuos en jeringas
residuo_DC1=(1-(TOTAL_DC1/105200)%1)*105200
if residuo_DC1 == 105200.0:
    residuo_DC1 = 0
residuo_DC2=(1-(TOTAL_DC2/105200)%1)*105200
if residuo_DC2 == 105200.0:
    residuo_DC2=0
residuo_total=(1-(TOTAL_DEMANDA/105200)%1)*105200

# residuo directo de DC2 sin contar pedaceria de sobra
residuo_norm_dc2=((TOTAL_DC2/105200)%1)*105200


#generación de reporte 
print("")
print("Reporte después del balanceo")
print("TOTAL_DC1: "+str(TOTAL_DC1)+" total jeringas")
print("TOTAL_DC2: "+str(TOTAL_DC2)+" total jeringas")
print("TOTAL_DEMANDA: "+str(TOTAL_DEMANDA)+" total jeringas")
print("Lotes DC1 "+str(LOTES_DC1))
print("Lotes DC2 "+str(LOTES_DC2))
print("Total Demanda Lotes "+str(lotes_necesarios))
print("Residuo DC1 "+str(residuo_DC1)+" jeringas de sobra")
print("Residuo DC2 "+str(residuo_DC2)+" jeringas de sobra")
print("residuo de la demanda total "+str(residuo_DC1+residuo_DC2)+" jeringas de sobra")
print("Residuo Total "+str(residuo_total)+" jeringas de sobra")